# NextGenUp — Google Colab

Run NextGenUp (AI image/video studio) on Google Colab with a public ngrok URL.

Run cells in order.

In [1]:
!nvidia-smi 2>/dev/null || echo 'No GPU (CPU mode is fine)'

Sat Oct 31 09:37:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null 2>&1
!ffmpeg -version | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [3]:
import os, shutil, urllib.request
REPO_URL = "https://github.com/riponcm/nextgenUp.git"
DEST = "/content/nextgenUp"
if os.path.exists(DEST):
    shutil.rmtree(DEST)
!git clone "$REPO_URL" "$DEST"
print("Cloned. Files:", os.listdir(DEST))

Cloning into '/content/nextgenUp'...
remote: Enumerating objects: 552, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 552 (delta 51), reused 88 (delta 37), pack-reused 438 (from 1)
Receiving objects: 100% (552/552), 9.11 MiB | 5.71 MiB/s, done.
Resolving deltas: 100% (251/251), done.
Cloned. Files: ['setup.sh', 'bg_remove.py', 'src-tauri', 'tests', 'audio_engine.py', 'README.md', 'templates', 'package.json', 'llms.txt', '.gitignore', 'assets', 'video_tools.py', 'userfiles.py', '.github', 'ai_engine.py', 'nextgenup-server.spec', 'LICENSE', 'restore_engine.py', 'model_store.py', 'paths.py', 'static', 'face_restore.py', '.git', 'app.py', 'desktop-ui', 'requirements.txt']


In [4]:
!pip install -q -r /content/nextgenUp/requirements.txt
print("Dependencies installed")

Dependencies installed


In [5]:
import os
os.makedirs("/content/nextgenUp/static/models", exist_ok=True)
model = "/content/nextgenUp/static/models/realesr-general-x4v3.onnx"
if not os.path.exists(model):
    print("Downloading Real-ESRGAN ONNX model (~5MB)...")
    urllib.request.urlretrieve(
        "https://huggingface.co/OwlMaster/AllFilesRope/resolve/main/realesr-general-x4v3.onnx",
        model
    )
    print("Model downloaded.")
else:
    print("Model already exists.")

Model downloaded.


In [6]:
from google.colab import userdata
import os
os.environ['NGROK_TOKEN'] = userdata.get('NGROK_TOKEN')
print("NGROK_TOKEN loaded")

NGROK_TOKEN loaded


In [11]:

import os, threading, time
from pyngrok import ngrok

token = "2mYxC0yBuZ6Tx674mLR9tud7mC1_Eyba45XFKhHTfQR6hPpf"
if not token:
    print('ERROR: NGROK_TOKEN not set')
else:
    ngrok.set_auth_token(token)
    # start app.py in background
    os.chdir('/content/nextgenUp')
    import subprocess
    proc = subprocess.Popen(
        ['python', 'app.py'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    time.sleep(5)
    public_url = ngrok.connect(8000, 'http').public_url
    print('=' * 60)
    print(f'NextGenUp URL: {public_url}')
    print('=' * 60)
    print('Open the URL above in your browser.')
    print('Keep this cell running.')
    # keep alive
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print('Shutting down...')
        ngrok.kill()
        proc.terminate()

PyngrokNgrokError: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'